<font color="#CA0032"><h1 align="left">**Redes recurrentes profundas**</h1></font>

<font color="#6E6E6E"><h1 align="left">**Predicción de series temporales**</h1></font>

<h2 align="left">Manuel Sánchez-Montañés</h2>

<font color="#6E6E6E"><h2 align="left">manuel.smontanes@gmail.com</h2></font>

**Notebook: Manuel Sánchez-Montañés**

**Datos: Carlos Rosado**

### **Usaremos un esquema many to one:**

<img src="https://drive.google.com/uc?export=download&id=1iokh576AiK2iFhftPogSBsNXixAi-LBg" align="center" style="float" width="500">

In [ ]:
COLAB = True

## <font color="#CA3532"> **Importar librerías**

In [ ]:
import numpy as np
import pandas as pd

from keras.models import Sequential, load_model
from keras.layers import Dense, LSTM, GRU
from keras.callbacks import ModelCheckpoint

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score as R2_score

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

In [ ]:
def download_file_from_google_drive(file_id, dest_file, unzip=False):
  aux = "'https://drive.usercontent.google.com/download?id={}&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280'".format(file_id)
  !wget $aux -O $dest_file
  if unzip:
    !unzip -qq -o $dest_file
    !rm $dest_file

## <font color="#CA3532"> **Carga de datos**

In [ ]:
if COLAB:
    download_file_from_google_drive(file_id='12QZpA_L1JncFIVEryee0aeL66Ep3xpOy', dest_file='./datos_pasajeros.csv')

data = pd.read_csv('datos_pasajeros.csv')
data.head(20)

## <font color="#CA3532"> **Librerías y funciones adicionales**

In [ ]:
if COLAB:
    download_file_from_google_drive(file_id='1LYuVxpFdsoxgl89tku6BtEH3HuYcGd2g',
                                    dest_file='./my_utils_series_temporales.py.zip', unzip=True)

In [ ]:
from my_utils_series_temporales import int2dummy, enventanar, info_enventanado, NAN

In [ ]:
def grafica_entrenamiento(tr_mse, val_mse):
    ax=plt.figure(figsize=(10,4)).gca()
    plt.plot(1+np.arange(len(tr_mse)), tr_mse)
    plt.plot(1+np.arange(len(val_mse)), val_mse)
    plt.title('mse del modelo', fontsize=18)
    plt.xlabel('epoca', fontsize=18)
    plt.ylabel('mse', fontsize=18)
    plt.legend(['entrenamiento', 'validación'], loc='upper left')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.show()

## <font color="#CA3532"> **Desarrollo y validación del modelo**

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data["fecha"][0]

In [ ]:
data.set_index("fecha", inplace=True)

In [ ]:
data.head()

In [ ]:
fechas      = data.index.values
target      = data["npasajeros"].values
festivo     = data["festivo"].values
mes         = data["mes"].values
semana_mes  = data["semana_mes"].values
day_of_week = data["day_of_week"].values

In [ ]:
data.describe()

In [ ]:
mes = mes - 1 # para que empiece en 0 (luego asociaremos un embedding a mes, y el código de mes tiene que empezar en 0)

In [ ]:
data["npasajeros"].plot(figsize=(15,5));

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(data["npasajeros"][:56], "-o");

In [ ]:
data[["mes", "semana_mes", "day_of_week"]].plot(figsize=(15,5));

In [ ]:
fecha_corte = data.index[int(0.8*len(data))]
fecha_corte

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(data[["npasajeros"]][:fecha_corte])

In [ ]:
target_transf = scaler.transform(data[["npasajeros"]])[:,0]
target_transf[:5]

In [ ]:
# prueba enventanado

# series univariables a enventanar:
series = [target_transf, festivo, day_of_week]
se_saben_antes = [False, True, False]
nombres_series = ["target_transf", "festivo", "day_of_week"]

data_window = 5 # lookback (tamaño de la ventana)

X, y = enventanar(series,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes, # si adelanto o no 1 día la información
                  W_in=data_window)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
X[5]

In [ ]:
info_enventanado(X[:10], y[:10], nombres_series,
                 nombre_target="target_transf", tiempos=fechas)

In [ ]:
# Lo anterior era una prueba. Ahora voy a enventanar cada
# variable por separado, porque las voy a tratar por separado
# en mi red

series0         = [target_transf]
se_saben_antes0 = [False]
nombres_series0 = ["target_transf"]

X0, y = enventanar(series0,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes0,
                  W_in=data_window)


series1         = [festivo]
se_saben_antes1 = [True]
nombres_series1 = ["festivo"]

X1, _ = enventanar(series1,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes1,
                  W_in=data_window)

series2         = [mes]
se_saben_antes2 = [True]
nombres_series2 = ["mes"]

X2, _ = enventanar(series2,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes2,
                  W_in=data_window)

series3         = [semana_mes]
se_saben_antes3 = [True]
nombres_series3 = ["semana_mes"]

X3, _ = enventanar(series3,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes3,
                  W_in=data_window)

series4         = [day_of_week]
se_saben_antes4 = [True]
nombres_series4 = ["day_of_week"]

X4, _ = enventanar(series4,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes4,
                  W_in=data_window)

In [ ]:
X0.shape, X1.shape, X2.shape, X3.shape, X4.shape

In [ ]:
# posición de le fecha de corte dentro de data:
fcorte_pos = data.index.to_list().index(fecha_corte)
fcorte_pos

In [ ]:
target_train = target[data_window:fcorte_pos] # número de pasajeros
y_train      = y[data_window:fcorte_pos] # número de pasajeros estandarizados
fechas_train = fechas[data_window:fcorte_pos]

X0_train = X0[data_window:fcorte_pos]
X1_train = X1[data_window:fcorte_pos]
X2_train = X2[data_window:fcorte_pos]
X3_train = X3[data_window:fcorte_pos]
X4_train = X4[data_window:fcorte_pos]

In [ ]:
target_test = target[fcorte_pos:] # número de pasajeros
y_test      = y[fcorte_pos:] # número de pasajeros estandarizados
fechas_test = fechas[fcorte_pos:]

X0_test = X0[fcorte_pos:]
X1_test = X1[fcorte_pos:]
X2_test = X2[fcorte_pos:]
X3_test = X3[fcorte_pos:]
X4_test = X4[fcorte_pos:]

In [ ]:
X0_train.shape, X1_train.shape, X2_train.shape, X3_train.shape, X4_train.shape

In [ ]:
X0_test.shape, X1_test.shape, X2_test.shape, X3_test.shape, X4_test.shape

In [ ]:
627+158+5

In [ ]:
fecha_corte

In [ ]:
# Modelo

from keras.layers import Input, Embedding, concatenate
from keras import Model, optimizers # modo funcional

In [ ]:
dim_embedding = 2

input0 = Input(shape=(data_window, 1), name="endogena")
input1 = Input(shape=(data_window, 1), name="festivo")
input2 = Input(shape=(data_window,), name="mes") # mes es una categórica (índice)
input3 = Input(shape=(data_window,), name="semana_mes") # es una categórica (índice)
input4 = Input(shape=(data_window,), name="day_of_week") # es una categórica (índice)

embedding2 = Embedding(input_dim=12, # número de categorías
                       output_dim=dim_embedding,
                       input_length=data_window,
                       name="embedding_mes")(input2)

embedding3 = Embedding(input_dim=3, # número de categorías
                       output_dim=dim_embedding,
                       input_length=data_window,
                       name="embedding_semana_mes")(input3)

embedding4 = Embedding(input_dim=7, # número de categorías
                       output_dim=dim_embedding,
                       input_length=data_window,
                       name="embedding_day_of_week")(input4)

x = concatenate([input0, input1, embedding2, embedding3, embedding4])
x2 = LSTM(units=2)(x)
x3 = Dense(units=1)(x2)

model = Model(inputs=[input0, input1, input2, input3, input4],
              outputs=x3)
model.compile(loss="mean_squared_error", optimizer=optimizers.RMSprop())

In [ ]:
model.summary()

In [ ]:
from IPython.display import SVG
from keras.utils import model_to_dot

if COLAB:
  display(SVG(model_to_dot(model, show_shapes=True, dpi=72).create(prog="dot",
                                                                   format="svg")))
else:
  display(SVG(model_to_dot(model, show_shapes=True).create(prog="dot",
                                                           format="svg")))

In [ ]:
epochs = 2000
batch_size = 32
Nval = 200 # los 200 últimos días de training los voy a usar como validación

X0_tr = X0_train[:-Nval]
X1_tr = X1_train[:-Nval]
X2_tr = X2_train[:-Nval]
X3_tr = X3_train[:-Nval]
X4_tr = X4_train[:-Nval]
y_tr  = y_train[:-Nval]

X0_va = X0_train[-Nval:]
X1_va = X1_train[-Nval:]
X2_va = X2_train[-Nval:]
X3_va = X3_train[-Nval:]
X4_va = X4_train[-Nval:]
y_va  = y_train[-Nval:]

In [ ]:
tr_loss_history = []
va_loss_history = []

modelpath = "best_model.h5" # fichero en el que guardaré el mejor modelo
checkpoint = ModelCheckpoint(modelpath, monitor="val_loss", verbose=2, save_best_only=True)

Xs_tr = {
    "endogena": X0_tr,
    "festivo": X1_tr,
    "mes": X2_tr,
    "semana_mes": X3_tr,
    "day_of_week": X4_tr,
}

Xs_va = {
    "endogena": X0_va,
    "festivo": X1_va,
    "mes": X2_va,
    "semana_mes": X3_va,
    "day_of_week": X4_va,
}

Xs_test = {
    "endogena": X0_test,
    "festivo": X1_test,
    "mes": X2_test,
    "semana_mes": X3_test,
    "day_of_week": X4_test,
}

In [ ]:
for e in range(epochs):
  history = model.fit(Xs_tr, y_tr, batch_size=batch_size, epochs=1,
                      callbacks=[checkpoint], verbose=0, validation_data=(Xs_va, y_va))
  tr_loss_history += history.history["loss"]
  va_loss_history += history.history["val_loss"]
  if e%50 == 0:
    grafica_entrenamiento(tr_loss_history, va_loss_history)

In [ ]:
model = load_model(modelpath)

In [ ]:
y_tr_prediction = model.predict(Xs_tr, verbose=0)

In [ ]:
y_va_prediction = model.predict(Xs_va, verbose=0)

In [ ]:
y_test_prediction = model.predict(Xs_test, verbose=0)

In [ ]:
target_tr_pred = scaler.inverse_transform(y_tr_prediction).flatten() # des-estandarización de los pasajeros
target_va_pred = scaler.inverse_transform(y_va_prediction).flatten()
target_test_pred = scaler.inverse_transform(y_test_prediction).flatten()

In [ ]:
# antes separé y_train en y_tr e y_va (estandarizados),
# pero no hice lo mismo con target_train (sin estandarizar)
# Ahora lo hago:
fechas_tr = fechas_train[:len(y_tr)]
fechas_va = fechas_train[len(y_tr):]
target_tr = target_train[:len(y_tr)]
target_va = target_train[len(y_tr):]

In [ ]:
plt.figure(figsize=(15,7))
plt.plot(fechas_tr, target_tr, '--', label="training")
plt.plot(fechas_va, target_va, '--', label="val")
plt.plot(fechas_test, target_test, '--', label="test")
plt.plot(fechas_tr, target_tr_pred, '--', label="pred training")
plt.plot(fechas_va, target_va_pred, '--', label="pred val")
plt.plot(fechas_test, target_test_pred, '--', label="pred test")
plt.legend()
plt.title("Predicción 1-step");

In [ ]:
!cal 9 2016

In [ ]:
fechas_test[0]

In [ ]:
plt.figure(figsize=(15,7))
plt.plot(fechas_test, target_test, '--o', label="test")
plt.plot(fechas_test, target_test_pred, '--', label="pred test")
plt.legend()
plt.title("Predicción 1-step");

In [ ]:
# esto es lo que daría si el modelo no tuviera ni idea (predice mañana lo mismo que hoy, "MODELO PERSISTENTE A 1 DÍA")

plt.figure(figsize=(15,7))
plt.plot(fechas_test, target_test, '--', label="test")
plt.plot(fechas_test[1:], target_test_pred[:-1], '--', label="pred test")
plt.legend();

In [ ]:
R2_score(target_tr, target_tr_pred), R2_score(target_va, target_va_pred), R2_score(target_test, target_test_pred)

In [ ]:
from sklearn.metrics import mean_squared_error as MSE

MSE(target_tr, target_tr_pred), MSE(target_va, target_va_pred), MSE(target_test, target_test_pred)

In [ ]:
1 - MSE(target_tr, target_tr_pred) / target_tr.var()

In [ ]:
# Extraigo métricas de diferentes modelos persistentes a N días:

R2_persistentes = []
rango = range(1, 4*7+1)
for i in rango:
  score = R2_score(target_test[i:], target_test_pred[:-i])
  R2_persistentes.append(score)
  print("R2 en test de modelo persistente a {} días: {}".format(i, score))

plt.plot(rango, R2_persistentes)
plt.title("R2 modelos persistentes")
plt.xlabel("N días persistente");

In [ ]:
embeddings_mes = model.get_layer("embedding_mes").get_weights()[0]
embeddings_mes.shape

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(embeddings_mes[:,0], embeddings_mes[:,1], "o")
nombres = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
for coords,nombre in zip(embeddings_mes,nombres):
  plt.text(coords[0],coords[1],nombre)

In [ ]:
embeddings_day_of_week = model.get_layer("embedding_day_of_week").get_weights()[0]
embeddings_day_of_week.shape

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(embeddings_day_of_week[:,0], embeddings_day_of_week[:,1], "o")
nombres = ["lun","mar","mie","jue","vie","sab","dom"]
for coords,nombre in zip(embeddings_day_of_week,nombres):
  plt.text(coords[0],coords[1],nombre)

In [ ]:
embeddings_semana_mes = model.get_layer("embedding_semana_mes").get_weights()[0]
embeddings_semana_mes.shape

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(embeddings_semana_mes[:,0], embeddings_semana_mes[:,1], "o")
nombres = ["principios mes","mediados mes","finales mes"]
for coords,nombre in zip(embeddings_semana_mes,nombres):
  plt.text(coords[0],coords[1],nombre)

In [ ]:
def multistep(model, Xs): # el número de ventanas en Xs = número de días hacia delante que quiero predecir
  ventana_endogena = Xs["endogena"][:1].copy()
  salidas = []
  for i in range(len(Xs["festivo"])):
    z = model.predict( # predigo un solo día (día i)
        {
            "endogena": ventana_endogena,
            "festivo": Xs["festivo"][i:(i+1)],
            "mes": Xs["mes"][i:(i+1)],
            "semana_mes": Xs["semana_mes"][i:(i+1)],
            "day_of_week": Xs["day_of_week"][i:(i+1)]
        }, verbose=0
    )
    ventana_endogena[:-1] = ventana_endogena[1:] # desplazo un día la ventana
    ventana_endogena[-1]  = z[0][0] # predicción
    salidas.append(scaler.inverse_transform(z)[0,0])

  return salidas

In [ ]:
Xs_test["endogena"][0]

In [ ]:
Xs_test["endogena"][1]

In [ ]:
Xs_test["endogena"][0].shape

In [ ]:
sal = multistep(model, Xs_test)

In [ ]:
plt.figure(figsize=(15,7))
plt.plot(fechas_test, target_test, '--o', label="test")
plt.plot(fechas_test, sal, '--', label="pred test")
plt.legend()
plt.title("Predicción multi-step");

In [ ]:
R2_score(target_test[:14], sal[:14])

In [ ]:
import pickle
aux = {
    "estandarizador": scaler,
    "data_window": data_window
}
with open("datos_modelo.pkl", "wb") as f:
  pickle.dump(aux, f)

## Producción

In [ ]:
model = load_model(modelpath)

In [ ]:
import pickle
with open("datos_modelo.pkl", "rb") as f:
  aux = pickle.load(f)
aux

In [ ]:
scaler = aux["estandarizador"]
data_window = aux["data_window"]

In [ ]:
data2 = data[-30:].copy()
data2["npasajeros"][5:] = NAN
data2

In [ ]:
fechas      = data2.index.values
target      = data2["npasajeros"].values
festivo     = data2["festivo"].values
mes         = data2["mes"].values
semana_mes  = data2["semana_mes"].values
day_of_week = data2["day_of_week"].values
target_transf = scaler.transform(data2[["npasajeros"]]).flatten()

mes = mes - 1

In [ ]:
target_transf[:10]

In [ ]:
# Lo anterior era una prueba. Ahora voy a enventanar cada
# variable por separado, porque las voy a tratar por separado
# en mi red

series0         = [target_transf]
se_saben_antes0 = [False]
nombres_series0 = ["target_transf"]

X0, y = enventanar(series0,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes0,
                  W_in=data_window)


series1         = [festivo]
se_saben_antes1 = [True]
nombres_series1 = ["festivo"]

X1, _ = enventanar(series1,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes1,
                  W_in=data_window)

series2         = [mes]
se_saben_antes2 = [True]
nombres_series2 = ["mes"]

X2, _ = enventanar(series2,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes2,
                  W_in=data_window)

series3         = [semana_mes]
se_saben_antes3 = [True]
nombres_series3 = ["semana_mes"]

X3, _ = enventanar(series3,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes3,
                  W_in=data_window)

series4         = [day_of_week]
se_saben_antes4 = [True]
nombres_series4 = ["day_of_week"]

X4, _ = enventanar(series4,
                  target=0, # cuál es la variable endógena a predecir
                  se_saben_antes=se_saben_antes4,
                  W_in=data_window)

In [ ]:
Xs_prod = {
    "endogena": X0[data_window:],
    "festivo": X1[data_window:],
    "mes": X2[data_window:],
    "semana_mes": X3[data_window:],
    "day_of_week": X4[data_window:],
}

In [ ]:
sal = multistep(model, Xs_prod)

In [ ]:
sal

In [ ]:
data2["prediccion"] = NAN
data2["prediccion"].iloc[data_window:] = sal

In [ ]:
data2

In [ ]:
data2["reales"] = data["npasajeros"][-30:].copy()
data2

In [ ]:
data2[["prediccion", "reales"]].plot();